# ch15 练习：多级索引

与讲解版逐 Cell 对应，`assert` 验收保留。卡住回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 数据探索 | 多级交叉查询 |
| 数据准备及处理 | `groupby` 多键聚合 + `xs` 截面 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

dev = pd.read_csv(DATA / "device_defects.csv")

KEY = "台区编号"
CAT = "设备类型"
VAL = "负荷值"
IDX = pd.IndexSlice

print("pandas", pd.__version__)
print("dev", dev.shape, "| 台区", dev[KEY].nunique(), "| 类型", dev[CAT].nunique())

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


PAIR = ("STATION_A_01", "断路器")     # 有数据的组合（A_01×变压器 的负荷值全缺）
BAD_PAIR = ("STATION_A_01", "变压器")  # 全缺组合：均值 NaN、条数 0

mi = dev.set_index([KEY, CAT]).sort_index()          # 排好序的多级索引 (270, 11)
unsorted_mi = dev.set_index([KEY, CAT])              # 原始行序（未 lexsorted）
m = dev.groupby([KEY, CAT])[VAL].mean()              # groupby 两键 → MultiIndex Series

print("mi", mi.shape, "| 已排序:", mi.index.is_monotonic_increasing)
print("unsorted_mi 已排序:", unsorted_mi.index.is_monotonic_increasing)
print("m", m.shape)

## 题 1：groupby 两键的产物（对应讲解 §3.1）

In [ ]:
# TODO(1)：groupby 两键算负荷均值已给出（m）；请打印它的 nlevels、names、
#            并用 .loc 取 PAIR 点的值存入 v_pair
# 提示：m.index.nlevels ；m.loc[PAIR]
____
v_pair = ____

print("PAIR 均值:", round(float(v_pair), 4))
assert m.index.nlevels == 2 and m.index.names == [KEY, CAT]
assert len(m) == 88
assert round(float(v_pair), 4) == 171.24

## 题 2：as_index=False 的平面表（对应讲解 §3.1）

In [ ]:
# TODO(2)：groupby 两键、as_index=False、负荷均值，存入 m2
m2 = ____

print(m2.shape, m2.columns.tolist())
assert m2.shape == (88, 3)
assert m2.columns.tolist() == [KEY, CAT, VAL]

## 题 3：UnsortedIndexError 现场（对应讲解 §3.3）

In [ ]:
# TODO(3)：用 probe 测未排序的 unsorted_mi 做范围切片（结果存入 status, msg）
# 提示：unsorted_mi.loc[(slice("STATION_A_01"), slice("断路器", "变压器")), :]
status, msg = ____

# TODO(4)：未排序单点 loc[PAIR] 取负荷值列，均值存入 mean_unsorted
mean_unsorted = ____

print("未排序范围切片 →", status)
print("未排序单点均值:", round(float(mean_unsorted.mean()), 4))
assert status == "err" and msg == "UnsortedIndexError"
assert round(float(mean_unsorted.mean()), 4) == 171.24

## 题 4：IndexSlice 三板斧（对应讲解 §3.4）

In [ ]:
# TODO(5)：用 IDX 取「全部台区 × 变压器」的行，存入 sub2
# 提示：mi.loc[IDX[:, "变压器"], :]
sub2 = ____

# TODO(6)：取 A_01~C_01 台区、(变压器, 断路器) 两种类型的负荷值，存入 sub3
# 提示：IDX["STATION_A_01":"STATION_C_01", ["变压器", "断路器"]]
sub3 = ____

print("全部×变压器:", sub2.shape, "| A~C×两类:", sub3.shape)
assert sub2.shape == (55, 11)
assert len(sub3) == 36
assert round(float(sub3.mean()), 4) == 359.347

## 题 5：xs 截面（对应讲解 §3.5）

In [ ]:
# TODO(7)：xs 取第 1 层「变压器」的截面（默认 drop_level），存入 x2；
#            再取 drop_level=False 版本存入 x4
# 提示：mi.xs("变压器", level=1) ；drop_level 参数
x2 = ____
x4 = ____

print("x2:", x2.shape, x2.index.name, "| x4:", x4.shape, x4.index.names)
assert x2.shape == (55, 11) and x2.index.name == KEY
assert x4.shape == (55, 11) and x4.index.names == [KEY, CAT]

## 题 6：swaplevel 与重排（对应讲解 §3.6）

In [ ]:
# TODO(8)：对 mi 换层存入 sw；再排序存入 sw2；按新第 0 层取「变压器」存入 by_cat
sw = ____
sw2 = ____
by_cat = ____

print("swap names:", sw.index.names, "| 已排序:", sw.index.is_monotonic_increasing)
print("排序后取变压器:", by_cat.shape)
assert sw.index.names == [CAT, KEY]
assert not bool(sw.index.is_monotonic_increasing)
assert bool(sw2.index.is_monotonic_increasing)
assert len(by_cat) == 55

## 题 7：交付口径（对应讲解 §3.7）

In [ ]:
# TODO(9)：对 mi 丢掉第 1 层存入 d1；全部还原成列存入 r
d1 = ____
r = ____

print("droplevel:", d1.shape, d1.index.name, "| reset:", r.shape)
assert d1.shape == (270, 11) and d1.index.name == KEY
assert r.shape == (270, 13) and r.columns.tolist()[:2] == [KEY, CAT]

## 题 8（综合）：全缺组合识别 + unstack（对应讲解 §3.8-3.9）

In [ ]:
# TODO(10)：两键命名聚合（均值/条数）存入 g；判断 BAD_PAIR 的均值是否为 NaN
#            存入 bad_is_nan、条数存入 bad_count；最后 m.unstack(CAT) 存入 w
# 提示：g.agg(均值=(VAL, "mean"), 条数=(VAL, "count")) ；pd.isna(...)
g = ____
bad_is_nan = ____
bad_count = ____
w = ____

print("g:", g.shape, "| 全缺组合 NaN:", bad_is_nan, "| 条数:", bad_count)
print("unstack:", w.shape)
assert g.shape == (88, 2) and g.columns.tolist() == ["均值", "条数"]
assert bool(bad_is_nan) and int(bad_count) == 0
assert w.shape == (18, 5)
assert round(float(w.loc["STATION_A_01", "断路器"]), 4) == 171.24

---

## 综合自查

1. 题 3 里"单点能用、切片报错"的根因是什么？
2. 题 5 的 x2 和 x4 差在哪一层？什么时候要 `drop_level=False`？
3. 题 6 换层后为什么必须重新排序？
4. 题 8 的全缺组合与"组合不存在"怎么区分？

全部答得上来，进入 ch16（时间序列）。